In [1]:
import os
import re
import joblib
import pandas as pd
import numpy as np

import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Path relative to the notebooks folder
data_path = os.path.join("..", "data", "IMDB Dataset.csv")

df = pd.read_csv(data_path)

# Verify dimensions and check label balance
print(f"Dataset shape: {df.shape}")
print("\nClass distribution:")
print(df['sentiment'].value_counts())
df.head()

Dataset shape: (50000, 2)

Class distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Negations are critical for sentiment
stop_words.discard('not')
stop_words.discard('no')

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)           # Strip HTML tags
    text = re.sub(r'[^a-z\s]', '', text)         # Strip numbers and punctuation
    tokens = [w for w in text.split() if w not in stop_words]
    return ' '.join(tokens)

print("Cleaning text (this may take 15-30 seconds for 50k rows)...")
df['clean_review'] = df['review'].apply(clean_text)

# Convert labels to integers: 1 for positive, 0 for negative
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
print("Done cleaning!")
df[['clean_review', 'label']].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


Cleaning text (this may take 15-30 seconds for 50k rows)...
Done cleaning!


,clean_review,label
0,one reviewers mentioned watching oz episode yo...,1
1,wonderful little production filming technique ...,1
2,thought wonderful way spend time hot summer we...,1
3,basically theres family little boy jake thinks...,0
4,petter matteis love time money visually stunni...,1


In [4]:
X = df['clean_review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f"Training samples: {len(X_train):,}")
print(f"Testing samples:  {len(X_test):,}")

Training samples: 40,000
Testing samples:  10,000


In [5]:
# Vectorizer setup: 5,000 top unigrams and bigrams
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# 1. Logistic Regression Pipeline
lr_pipeline = Pipeline([
    ('tfidf', tfidf),
    ('classifier', LogisticRegression(max_iter=1000))
])

# 2. Naive Bayes Pipeline
nb_pipeline = Pipeline([
    ('tfidf', tfidf),
    ('classifier', MultinomialNB())
])

print("Training Logistic Regression...")
lr_pipeline.fit(X_train, y_train)

print("Training Naive Bayes...")
nb_pipeline.fit(X_train, y_train)

print("Training complete!")

Training Logistic Regression...
Training Naive Bayes...
Training complete!


In [6]:
lr_preds = lr_pipeline.predict(X_test)
nb_preds = nb_pipeline.predict(X_test)

print(f"Logistic Regression Accuracy: {accuracy_score(y_test, lr_preds):.4f}")
print(f"Naive Bayes Accuracy:         {accuracy_score(y_test, nb_preds):.4f}")

print("\n--- Logistic Regression Classification Report ---")
print(classification_report(y_test, lr_preds, target_names=['Negative', 'Positive']))

Logistic Regression Accuracy: 0.8937
Naive Bayes Accuracy:         0.8630

--- Logistic Regression Classification Report ---
              precision    recall  f1-score   support

    Negative       0.90      0.89      0.89      5000
    Positive       0.89      0.90      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [7]:
os.makedirs("../models", exist_ok=True)
model_file = os.path.join("..", "models", "sentiment_model.pkl")

joblib.dump(lr_pipeline, model_file)
print(f"Model saved to {model_file}")

Model saved to ..\models\sentiment_model.pkl


In [8]:
# Load saved pipeline
loaded_model = joblib.load(model_file)

test_sentences = [
    "The plot was slow and the acting felt very forced.",
    "Absolutely breathtaking cinematography and a brilliant script!",
    "It wasn't good at all.",
    "Not the worst thing I've watched, but definitely not great."
]

predictions = loaded_model.predict(test_sentences)
probabilities = loaded_model.predict_proba(test_sentences)

for text, pred, proba in zip(test_sentences, predictions, probabilities):
    label = "POSITIVE" if pred == 1 else "NEGATIVE"
    confidence = proba[pred] * 100
    print(f"[{label} ({confidence:.1f}%)] -> \"{text}\"")

[NEGATIVE (94.8%)] -> "The plot was slow and the acting felt very forced."
[POSITIVE (73.9%)] -> "Absolutely breathtaking cinematography and a brilliant script!"
[POSITIVE (90.9%)] -> "It wasn't good at all."
[NEGATIVE (75.6%)] -> "Not the worst thing I've watched, but definitely not great."
